# Pasado `gpu-hybrid`: poređenje GPU zamena za `gelsd` na CUDA-i

Pre pokretanja izaberi **Runtime > Change runtime type > GPU (A100)**, pa **Runtime > Run all**.

Pravilo je pre-registrovano u `experiments/compare_lstsq_solvers.py` (commit `385397e`) i ova sveska ga ne menja. Na Mac-u su već urađeni CPU runovi v1 i v2 (`results/lstsq_comparison_2026-10-25*`): u v2 su A2 i H izjednačeni posle R3, a odluku daje **R4, brzina solvera na A100**.

Šta sveska radi:
1. Proveri GPU (pravilo meri R4 na **A100**) i klonira `gpu-hybrid` na fiksnom commitu `7740ba4`.
2. **Smoke na CUDA-i**: sve četiri putanje (box, network, timing, decide) na par slučajeva. Nije merenje.
3. **Timing (R4)**: svaki kandidat na `n = 1024, 8192, 32768, 65536`, `float64` i `float32`, `timing_cuda.csv`.
4. **Box nivo na CUDA-i**: G0 + self-test sa 10 zasađenih bagova (ispravljen posle prvog CUDA runa, u kom je S4 preživeo), pa svih 12 familija. Ovde se P prvi put ocenjuje na cuSOLVER `gesvdj`, umesto na CPU `gesdd`.
5. **Network nivo na CUDA-i**: samo preživeli posle G0 do G2 (automatski).
6. **`decide`** nad CUDA rezultatima.
7. **Informativno, nije deo pravila**: ceo batched prolaz na `big` sa `gelsd` naspram A2 i H, `n_splits = 1, 8, 32, 64`.
8. Zip sa svim rezultatima za preuzimanje.

Očekivano trajanje: oko 20 do 40 minuta (oracle i mpmath provere u box nivou rade na CPU-u Colab mašine).
Kad završi, pošalji `lstsq_comparison_cuda.zip` (preuzima se sam u poslednjoj ćeliji).

In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
assert torch.cuda.is_available(), "Uključi GPU: Runtime > Change runtime type > GPU"

GPU = torch.cuda.get_device_name(0)
VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
print("Torch:", torch.__version__, "| CUDA:", torch.version.cuda, "| GPU:", GPU,
      "| VRAM GiB:", round(VRAM_GIB, 1))

# R4 of the rule is defined on the A100. Set to False only for a dry run; the
# timing is then not valid for R4.
REQUIRE_A100 = True
assert "A100" in GPU or not REQUIRE_A100, (
    f"Pravilo meri R4 na A100, a ovo je {GPU}. Promeni runtime "
    "ili postavi REQUIRE_A100 = False (tada R4 ne važi).")

In [ ]:
import os
import shutil

REPO = "/content/Pasado"
URL = "https://github.com/dimitrijepesic/Pasado.git"
COMMIT = "7740ba4"  # v2 candidates, rule of 385397e with the self-test of 67f4654

# leave the folder first: deleting the current directory breaks git on a re-run
os.chdir("/content")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "-c", "http.version=HTTP/1.1", "clone", "--branch", "gpu-hybrid",
                "--single-branch", URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert head.startswith(COMMIT), f"HEAD je {head}, a ocekivan je {COMMIT}"
print("HEAD:", head)
os.chdir(REPO)

In [ ]:
%pip install -q scikit-learn mpmath

In [ ]:
OUT = "/content/lstsq_comparison_cuda"
os.makedirs(OUT, exist_ok=True)


def run(args, log):
    """compare_lstsq_solvers.py with its output on screen and in a log file."""
    cmd = f'''
set -o pipefail
PYTHONUNBUFFERED=1 python experiments/compare_lstsq_solvers.py {args} 2>&1 | tee {log}
'''
    return subprocess.run(["bash", "-lc", cmd], cwd=REPO).returncode

## 1. Smoke na CUDA-i

Proverava da sve četiri putanje rade na GPU-u, sa malim brojem slučajeva i drugim seed-ovima. **Nije merenje.** Smoke uvek završava sa `Undecided at R4` i self-test na ovako malom uzorku može da prijavi preživeli mutant (`smoke run: continuing`); to je očekivano.

In [ ]:
rc = run("--smoke --device cuda --out /content/smoke_cuda", "/content/smoke_cuda.log")
assert rc == 0, "smoke na CUDA-i nije prošao; ne nastavljaj, pošalji /content/smoke_cuda.log"

## 2. Timing (R4)

Medijana od 10 ponavljanja posle 3 warm-up-a, sa `torch.cuda.synchronize()` oko svakog. G uključuje prenos na CPU i nazad, kao u produkciji. R4 koristi `float64`, `n = 65536`.

In [ ]:
rc = run(f"--level timing --device cuda --out {OUT}", f"{OUT}/timing_level.log")
assert rc == 0, "timing nije prošao; pošalji log"

import pandas as pd
from IPython.display import display

t = pd.read_csv(f"{OUT}/timing_cuda.csv")
for dt in ("float64", "float32"):
    print(dt, "(median ms)")
    display(t[t.dtype == dt].pivot(index="candidate", columns="n", values="median_ms").round(3))

## 3. Box nivo na CUDA-i

Isti skup slučajeva kao na Mac-u (seed-ovi, 12 familija, pravi boxovi sa 4 mreže), ali solveri rade na GPU-u. Ako self-test ne uhvati neki zasađeni bag, run se zaustavlja (exit 2) pre merenja.

In [ ]:
rc = run(f"--level box --device cuda --out {OUT}", f"{OUT}/box_level.log")
assert rc == 0, "box nivo nije prošao (self-test ili greška); ne nastavljaj, pošalji box_level.log"

## 4. Network nivo na CUDA-i

Sam čita box rezultate iz istog foldera i pokreće samo kandidate koji su prošli G0 do G2 (plus G). 4 mreže, 3 slike, 5 epsilona, `n_splits = 1, 4, 16`.

In [ ]:
rc = run(f"--level network --device cuda --out {OUT}", f"{OUT}/network_level.log")
assert rc == 0, "network nivo nije prošao; pošalji network_level.log"

## 5. Odluka po pravilu (CUDA rezultati)

In [ ]:
rc = run(f"--level decide --out {OUT}", f"{OUT}/decide.log")

## 6. Informativno: ceo batched prolaz na `big`

Nije deo pravila. Meri koliko ceo batched prolaz na `big` traje sa `gelsd` (G) naspram A2 i H, za `n_splits = 1, 8, 32, 64` (1 slika, `eps_idx = 0`, seed 7, 1 warm-up + 3 ponavljanja). Za poređenje, sekvencijalna petlja na A100 je 24.10 trajala 4066.7 ms na `n_splits = 64`, a batched sa `gelsd` 2536.3 ms (`results/gpu_hybrid_batched_splits_2026-10-24/README.md`). Granice moraju da se slažu sa G-om do oko `1e-13`.

In [ ]:
import math
import statistics
import sys
import time

for p in (f"{REPO}/forward_mode_tensorized_src", f"{REPO}/Section_5_4", f"{REPO}/experiments"):
    if p not in sys.path:
        sys.path.insert(0, p)

import batched_splits as bsp
import lstsq_candidates as lc
from compare_lstsq_solvers import EPS_SCHEDULE, set_production_flags, swapped_solver
from measure_split_sweep import load_network_and_images

set_production_flags()
device = torch.device("cuda")
net, imgs = load_network_and_images("big", device, torch.float64, 1)
layers = bsp.extract_affine_layers(net, "big")
img = imgs[0][1]
eps = EPS_SCHEDULE[0]

# about 0.5 GB per split on big in float64; keep to 60% of VRAM
BIG_CHUNK = min(64, 2 ** int(math.log2(max(1, 0.6 * VRAM_GIB / 0.5))))
print(f"VRAM {VRAM_GIB:.1f} GiB -> chunk {BIG_CHUNK}")

e2e = []
for n_splits in (1, 8, 32, 64):
    for name in ("G", "A2", "H"):
        with swapped_solver(lc.CANDIDATES[name]):
            bsp.split_bounds(layers, img, eps, n_splits, seed=7, chunk=BIG_CHUNK)  # warm-up
            times = []
            for _ in range(3):
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                bound = bsp.lipschitz_bound(layers, img, eps, n_splits, seed=7, chunk=BIG_CHUNK)
                torch.cuda.synchronize()
                times.append((time.perf_counter() - t0) * 1000.0)
        e2e.append(dict(n_splits=n_splits, solver=name, median_ms=statistics.median(times),
                        bound=bound, chunk=BIG_CHUNK, gpu=GPU))
        print(f"n_splits {n_splits:3d} {name:2s} {statistics.median(times):9.1f} ms  bound {bound:.10g}")

e2e = pd.DataFrame(e2e)
ref = e2e[e2e.solver == "G"].set_index("n_splits")
e2e["speedup_vs_G"] = e2e.apply(lambda r: ref.loc[r.n_splits, "median_ms"] / r.median_ms, axis=1)
e2e["rel_diff_bound_vs_G"] = e2e.apply(
    lambda r: abs(r.bound - ref.loc[r.n_splits, "bound"]) / abs(ref.loc[r.n_splits, "bound"]), axis=1)
e2e.to_csv(f"{OUT}/e2e_big_batched.csv", index=False)
display(e2e.round({"median_ms": 1, "speedup_vs_G": 2}))

## 7. Preuzimanje rezultata (`.zip`)

In [ ]:
import json
from zipfile import ZIP_DEFLATED, ZipFile
from google.colab import files

with open(f"{OUT}/colab_environment.json", "w") as f:
    json.dump(dict(gpu=GPU, vram_gib=VRAM_GIB, torch=torch.__version__, cuda=torch.version.cuda,
                   commit=head), f, indent=2)

archive = "/content/lstsq_comparison_cuda.zip"
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for root, _, names in os.walk(OUT):
        for name in names:
            path = os.path.join(root, name)
            z.write(path, os.path.relpath(path, "/content"))
    z.write("/content/smoke_cuda.log", "smoke_cuda.log")
print(archive, round(os.path.getsize(archive) / 2**20, 1), "MiB")
files.download(archive)